# Lesson 3: Agentic Search

### 本节课的核心思路：什么是「Agentic Search」

前两课里 Agent 调用的搜索工具（Tavily）本身就是专门为 LLM 设计的「智能搜索」：不只是返回一堆链接，
还能直接给出 `include_answer=True` 这样结构化、精简过的答案，减少 Agent 还要自己抓网页、清洗 HTML 的工作量。

这一课用一个对比实验来说明为什么需要"agentic"（为 Agent 设计）的搜索工具：
1. 先用普通的 DuckDuckGo 搜索 + `requests` + `BeautifulSoup` 手动抓取、清洗网页内容（传统爬虫方式，噪音多、步骤繁琐）
2. 再用 Tavily 这种 agentic search 工具直接拿到干净的结果，对比两种方式的差异

这解释了为什么大多数生产级 Agent 会选择 Tavily 这类工具作为 `tool`，而不是自己写爬虫。

In [ ]:
from dotenv import load_dotenv
import os
from tavily import TavilyClient
_=load_dotenv()
client=TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))  # 创建 Tavily 客户端，构造时不发请求，第一次 .search() 才会真正调用 API

In [ ]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)  # include_answer=True：让 Tavily 直接总结出一个简短答案，而不只是返回搜索结果列表
# print the answer
result["answer"]

In [ ]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

In [ ]:
import requests
from bs4 import BeautifulSoup
# 提示：duckduckgo_search 这个包已改名为 ddgs（pip install ddgs），当前仍可用，只会有 RuntimeWarning，不影响运行
from duckduckgo_search import DDGS
import re

ddg=DDGS()
def search(query,max_results=6):
    try:
        results=ddg.text(query,max_results=max_results)  # 普通搜索：只返回 title/href/body，没有 Tavily 那种"直接给答案"的能力
        return [i ["href"] for i in results]              # 从结果列表里只取每条结果的链接
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results
for i in search(query):
    print(i)

In [ ]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."

    # fetch data
    headers={'User-Agent': 'Mozilla/5.0'}   # 伪装成浏览器 UA，避免部分网站直接拒绝无 UA 的请求
    response=requests.get(url,headers=headers)
    if response.status_code!=200:
        return "Failed to retrieve the webpage."
    soup=BeautifulSoup(response.text,'html.parser')   # 用 BeautifulSoup 把原始 HTML 解析成可查询的 DOM 树
    return soup

In [ ]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs  # 抓下来的是原始 HTML（含大量标签、样式、脚本噪音），下一步才会清洗成纯文本

In [ ]:
weather_data=[]
for tag in soup.find_all(['h1','h2','h3','p']):   # 只挑标题和段落标签，过滤掉 script/style/nav 等噪音标签
    text=tag.get_text(" ",strip=True)
    weather_data.append(text)
weather_data="\n".join(weather_data)
weather_data=re.sub(r'\s+',' ',weather_data)      # 把多个连续空白字符压缩成一个空格，让文本更紧凑
print(f"Website: {url}\n\n")
print(weather_data)
# 对比一下：光是拿到"干净文本"就需要 请求网页 -> 解析 HTML -> 按标签过滤 -> 正则清洗 这好几步，
# 而且不同网站的 HTML 结构完全不同，这段代码换个网站可能就要重写。这正是下面 Tavily 一行代码要解决的问题。

In [ ]:
result=client.search(query,max_results=1)      # 用 Tavily 搜同一个问题：一次调用直接拿到已经提炼好的网页内容
data=result["results"][0]['content']
print(data)

In [ ]:
import  json
from pygments import highlight,lexers, formatters
# 注意：这里用 data.replace("'",'"') 把单引号硬替换成双引号来凑成合法 JSON，只是应付课程演示的取巧写法，
# 如果 content 文本里本身含有英文撇号（比如 "it's"、"today's"）就会把 JSON 结构破坏掉导致 json.loads 报错，
# 生产代码里更稳妥的做法是用 ast.literal_eval 或者直接确认 Tavily 返回的就是合法 JSON 字符串。
parsed_json=json.loads(data.replace("'",'"'))
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())   # 终端高亮打印 JSON，方便阅读结构化结果

print(colorful_json)
